# **AVANCE_3: Data Warehouse y ETL**


FleetLogix - Modelado Dimensional y Pipeline ETL\
En el avance 3 se realiza la apertura de una cuenta en Snowflake y se crea el Warehouse. Después de crear las tablas 
de hechos y dimensiones, se poblan usando un pipeline conectado a Postgres y Snowflake. 

**Arquitectura de Datos**

1. Modelo OLTP: PostgreSQL: deliveries, trips, routes, vehicles, drivers, maintenance.
2. Procesamiento ETL: Python: Pandas, psycopg2, snowflake-connector-python, schedul.
3. Destino OLAP: Snowflake Data Warehouse: FLEETLOGIX_DW.ANALYTICS.

## **1. Data Warehouse en Snowflake**

El diseño del Data Warehouse es tipo Estrella compuesto por 1 tabla de hechos y 6 tablas de dimensiones.

Diagrama del Modelo Estrella:
* Tabla de Hechos: fact_deliveries
* Tablas de Dimensiones: dim_date, dim_time, dim_vehicle, dim_driver, dim_route, dim_customer.
* Tabla de Resumen Diario: daily_delivery_totals.

### **1.1. Creación Base de Datos**

In [ ]:
'''
-- =====================================================
-- FLEETLOGIX - DATA WAREHOUSE DIMENSIONAL MODEL
-- Modelo estrella para análisis en Snowflake
-- =====================================================

-- Crear warehouse y database en Snowflake
USE ROLE ACCOUNTADMIN;
CREATE WAREHOUSE IF NOT EXISTS FLEETLOGIX_WH WITH WAREHOUSE_SIZE = XSMALL AUTO_SUSPEND = 60;
CREATE DATABASE IF NOT EXISTS FLEETLOGIX_DW;
USE DATABASE FLEETLOGIX_DW;
CREATE SCHEMA IF NOT EXISTS ANALYTICS;
USE SCHEMA ANALYTICS;
'''

### **1.2. Creación Tablas de Dimensiones**

Son en total seis tablas que se crearán en Snowflake

dim_date:	    Permite granularidad en diferentes intervalos de tiempo\
dim_time:	    Permite granularidad a un nivel de: horas, minutos y segundos\ 
dim_vehicle:    Contiene los atributos del vehículo\
dim_driver	    Contiene los atributos del conductor\
dim_route	    Contiene los atributos de la ruta\
dim_customer	Contiene los atributos del cliente: se agregó en customer_key "AUTOINCREMENT START 1 INCREMENT 1 PRIMARY KEY"
                debido a que generaba error si se tenía solo así  customer_key INT PRIMARY KEY, ya que se generaban null y al ser una llave no puede ser null. 

In [ ]:
'''
-- =====================================================
-- DIMENSIONES
-- =====================================================

-- Dimensión Fecha
CREATE OR REPLACE TABLE dim_date (
    date_key INT PRIMARY KEY,
    full_date DATE NOT NULL,
    day_of_week INT,
    day_name VARCHAR(10),
    day_of_month INT,
    day_of_year INT,
    week_of_year INT,
    month_num INT,
    month_name VARCHAR(10),
    quarter INT,
    year INT,
    is_weekend BOOLEAN,
    is_holiday BOOLEAN,
    holiday_name VARCHAR(50),
    fiscal_quarter INT,
    fiscal_year INT
);

-- Dimensión Tiempo (para análisis por hora)
CREATE OR REPLACE TABLE dim_time (
    time_key INT PRIMARY KEY,
    hour INT,
    minute INT,
    second INT,
    time_of_day VARCHAR(20), -- 'Madrugada', 'Mañana', 'Tarde', 'Noche'
    hour_24 VARCHAR(5),       -- '14:30'
    hour_12 VARCHAR(8),       -- '02:30 PM'
    am_pm VARCHAR(2),
    is_business_hour BOOLEAN,
    shift VARCHAR(20)         -- 'Turno 1', 'Turno 2', 'Turno 3'
);

-- Dimensión Vehículo
CREATE OR REPLACE TABLE dim_vehicle (
    vehicle_key INT PRIMARY KEY,
    vehicle_id INT NOT NULL,
    license_plate VARCHAR(20),
    vehicle_type VARCHAR(50),
    capacity_kg DECIMAL(10,2),
    fuel_type VARCHAR(20),
    acquisition_date DATE,
    age_months INT,
    status VARCHAR(20),
    last_maintenance_date DATE,
    valid_from DATE,
    valid_to DATE,
    is_current BOOLEAN
);

-- Dimensión Conductor
CREATE OR REPLACE TABLE dim_driver (
    driver_key INT PRIMARY KEY,
    driver_id INT NOT NULL,
    employee_code VARCHAR(20),
    full_name VARCHAR(200),
    license_number VARCHAR(50),
    license_expiry DATE,
    phone VARCHAR(20),
    hire_date DATE,
    experience_months INT,
    status VARCHAR(20),
    performance_category VARCHAR(20), -- 'Alto', 'Medio', 'Bajo'
    valid_from DATE,
    valid_to DATE,
    is_current BOOLEAN
);

-- Dimensión Ruta
CREATE OR REPLACE TABLE dim_route (
    route_key INT PRIMARY KEY,
    route_id INT NOT NULL,
    route_code VARCHAR(20),
    origin_city VARCHAR(100),
    destination_city VARCHAR(100),
    distance_km DECIMAL(10,2),
    estimated_duration_hours DECIMAL(5,2),
    toll_cost DECIMAL(10,2),
    difficulty_level VARCHAR(20), -- 'Fácil', 'Medio', 'Difícil'
    route_type VARCHAR(20)        -- 'Urbana', 'Interurbana', 'Rural'
);

-- Dimensión Cliente
CREATE OR REPLACE TABLE dim_customer (
    customer_key INT AUTOINCREMENT START 1 INCREMENT 1 PRIMARY KEY,
    customer_name VARCHAR(200) NOT NULL,
    customer_type VARCHAR(50),
    city VARCHAR(100),
    first_delivery_date DATE,
    total_deliveries INT,
    customer_category VARCHAR(20)
);
'''

### **1.3. Creación Tabla de Hechos**

La Tabla de Hechos cuenta con los siguientes tipos de variables:\
1. keys: llaves primarias y foráneas
2. degenerate dimensions: id y códigos
3. métricas: medibles
4. métricas calculadas: se obtienen a partir de operaciones matemáticas
5. indicadores: booleanos 

In [ ]:
'''
-- =====================================================
-- TABLA DE HECHOS
-- =====================================================

CREATE OR REPLACE TABLE fact_deliveries (
    -- Keys
    delivery_key INT IDENTITY PRIMARY KEY,
    date_key INT REFERENCES 
dim_date(date_key),
    scheduled_time_key INT REFERENCES dim_time(time_key),
    delivered_time_key INT REFERENCES dim_time(time_key),
    vehicle_key INT REFERENCES dim_vehicle(vehicle_key),
    driver_key INT REFERENCES dim_driver(driver_key),
    route_key INT REFERENCES 
dim_route(route_key),
    customer_key INT REFERENCES dim_customer(customer_key),
    
    -- Degenerate dimensions
    delivery_id INT NOT NULL,
    trip_id INT NOT NULL,
    tracking_number VARCHAR(50),
    
    -- Métricas
    package_weight_kg DECIMAL(10,2),
    distance_km DECIMAL(10,2),
    fuel_consumed_liters DECIMAL(10,2),
    delivery_time_minutes INT,
    delay_minutes INT,
    
    -- Métricas calculadas
    deliveries_per_hour DECIMAL(5,2),
    fuel_efficiency_km_per_liter DECIMAL(5,2),
    cost_per_delivery DECIMAL(10,2),
    revenue_per_delivery DECIMAL(10,2),
    
    -- Indicadores
    is_on_time BOOLEAN,
    is_damaged BOOLEAN,
    has_signature BOOLEAN,
    delivery_status VARCHAR(20),
    
    -- Auditoría
    etl_batch_id INT,
    etl_timestamp TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);
'''

### **1.4. Configuración Snowflake**

Se habilita la recepción de datos en Snowflake, con un tiempo de 30 días\
Con Staging se genera la tabla de recepción de datos con registro del momento exacto de su carga.

In [ ]:
'''
-- =====================================================
-- CONFIGURACIÓN SNOWFLAKE
-- =====================================================

-- Habilitar Time Travel (30 días)
ALTER TABLE fact_deliveries SET DATA_RETENTION_TIME_IN_DAYS = 30;
ALTER TABLE dim_date SET DATA_RETENTION_TIME_IN_DAYS = 30;
ALTER TABLE dim_vehicle SET DATA_RETENTION_TIME_IN_DAYS = 30;
ALTER TABLE dim_driver SET DATA_RETENTION_TIME_IN_DAYS = 30;
ALTER TABLE dim_route SET DATA_RETENTION_TIME_IN_DAYS = 30;
ALTER TABLE dim_customer SET DATA_RETENTION_TIME_IN_DAYS = 30;
ALTER TABLE dim_time SET DATA_RETENTION_TIME_IN_DAYS = 30;

-- Crear tabla de staging para ETL
CREATE OR REPLACE TABLE staging_daily_load (
    raw_data VARIANT,
    load_timestamp TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);
'''

### **1.5. Creación de Vistas Seguras**

Se crean dos roles:\ 
1. Analista de ventas: acceso a métricas comerciales\ 
2. Analista de operaciones: no puede observar datos de facturación e ingresos.

In [ ]:
'''
-- =====================================================
-- VISTAS SEGURAS POR ROL
-- =====================================================

-- Vista para Ventas (solo sus clientes)
CREATE OR REPLACE SECURE VIEW v_sales_deliveries AS
SELECT 
    d.full_date,
    c.customer_name,
    c.customer_type,
    f.package_weight_kg,
    f.delivery_status,
    f.revenue_per_delivery
FROM fact_deliveries f
JOIN dim_date d ON f.date_key = d.date_key
JOIN dim_customer c ON f.customer_key = c.customer_key
WHERE c.customer_type != 'Gobierno'; -- Restricción ejemplo

-- Vista para Operaciones (todo)
CREATE OR REPLACE SECURE VIEW v_operations_deliveries AS
SELECT 
    d.full_date,
    t.hour_24 as hora,
    v.license_plate,
    dr.full_name as conductor,
    r.route_code,
    c.customer_name,
    f.delivery_time_minutes,
    f.delay_minutes,
    f.is_on_time,
    f.fuel_consumed_liters
FROM fact_deliveries f
JOIN dim_date d ON f.date_key = d.date_key
JOIN dim_time t ON f.scheduled_time_key = t.time_key
JOIN dim_vehicle v ON f.vehicle_key = v.vehicle_key
JOIN dim_driver dr ON f.driver_key = dr.driver_key
JOIN dim_route r ON f.route_key = r.route_key
JOIN dim_customer c ON f.customer_key = c.customer_key;

-- Crear roles
CREATE ROLE IF NOT EXISTS SALES_ANALYST;
CREATE ROLE IF NOT EXISTS OPERATIONS_ANALYST;

-- Asignar permisos
GRANT SELECT ON VIEW v_sales_deliveries TO ROLE SALES_ANALYST;
GRANT SELECT ON VIEW v_operations_deliveries TO ROLE OPERATIONS_ANALYST;
'''

## **2. Pipeline ETL en Python**

El pipeline  realiza las siguientes tareas principales:
1. Extracción con 'extract_daily_data': Lee registros de PostgreSQL a partir del modelo OLTP que contiene las tablas deliveries (d), trips (t) y routes (r).
2. Transformación 'transform_data': 
    Calcula tiempo de entrega 'delivery_time_minutes' y demoras 'delay_minutes'.\
    Calcula indicadores clave (KPIs): 'deliveries_per_hour', 'fuel_efficiency_km_per_liter', 'cost_per_delivery' y 'revenue_per_delivery'.\
    Realiza validaciones de calidad de datos (filtros de tiempos y pesos no válidos).
3. Carga 'load_dimensions' y 'load_facts': Actualiza dimensiones mediante 'MERGE' e inserta hechos en lote en Snowflake.
4. Agregación '_calculate_daily_totals': Precalcula métricas en la tabla 'daily_delivery_totals'.

### **2.1. Librerías**

In [ ]:
import psycopg2                                 # Conecta Python a PostgreSQL 
import snowflake.connector                      # Conecta Python con Snowflake
import pandas as pd                             # Manipulación de dataframes
import numpy as np                              # Manipulación de vectores y matrices
from datetime import datetime, timedelta        # Permite la gestión de fechas y horas
import logging                                  # Permite registro de eventos y monitoreo del sistema
import schedule                                 # Permite la ejecución automática de funciones de Python 
import time                                     # Trabaja con el tiempo del sistema
import json                                     # Procesa datos en formato JSON 
from typing import Dict, List, Tuple            # Permite gestionar tipos de datos de entrada y salida

### **2.2. Datos para Conexiones**

In [ ]:
# Configuración de logging: sirve para el manejo automatico del llenado de datos diario
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('etl_pipeline.log'),
        logging.StreamHandler()
    ]
)

# Configuración de conexiones
# 1. Conexión de Python con Postgres, fue necesario colocar la clave correspondiente a postgres en el equipo
POSTGRES_CONFIG = {
    'host': 'localhost',
    'database': 'fleetlogix',
    'user': 'postgres',
    'password': '305uNAMETA',
    'port': 5432
}

# 2. Conexión de Python con Snowflake, fue necesario colocar el usuario, clave y cuenta y el nombre del Warehouse creado,
#    la base de datos y tipo de esquema
SNOWFLAKE_CONFIG = {
    'user': 'DECORAL',
    'password': 'Z5VJRXFfmAE6pmz',
    'account': 'xkkscmd-lm65343',
    'warehouse': 'FLEETLOGIX_WH',
    'database': 'FLEETLOGIX_DW',
    'schema': 'ANALYTICS'
}

### **2.3. Conexiones**

In [ ]:
# Se crea la clase FleetLogixETL donde se realizará Extracción, Transformación y Carga de datos.

class FleetLogixETL:

    # La primera función inicializa los contadores en cero para la extracción de datos

    def __init__(self):
        self.pg_conn = None
        self.sf_conn = None
        self.batch_id = int(datetime.now().timestamp())
        self.metrics = {
            'records_extracted': 0,
            'records_transformed': 0,
            'records_loaded': 0,
            'errors': 0
        }

    # Establecer conexiones con PostgreSQL y Snowflake

    def connect_databases(self):

        try:
            # PostgreSQL
            self.pg_conn = psycopg2.connect(**POSTGRES_CONFIG)
            logging.info(" Conectado a PostgreSQL")
            
            # Snowflake
            self.sf_conn = snowflake.connector.connect(**SNOWFLAKE_CONFIG)
            logging.info(" Conectado a Snowflake")
            
            return True
        except Exception as e:
            logging.error(f" Error en conexión: {e}")
            return False

### **2.4. Extracción de Datos**

Tiene una de las modificaciones realizadas al código para que funcionará 

In [ ]:
# Función para extracción de datos desde Postgres

    def extract_daily_data(self) -> pd.DataFrame:
        """Extraer datos del día anterior de PostgreSQL"""
        logging.info(" Iniciando extracción de datos...")

# Se completó esta query con datos que se requerían para calcular las métricas. Toma datos de las tablas:
# deliveries (d), trips(t) y routes (r) del modelo OLTP generado en el primer avance del proyecto.
        query = """
        SELECT
            d.delivery_id, 
            d.trip_id, 
            d.tracking_number, 
            d.package_weight_kg,
            d.scheduled_datetime, 
            d.delivered_datetime, 
            d.delivery_status, 
            d.recipient_signature,
            d.customer_name,                    
            r.destination_city AS destination_city, 
            t.departure_datetime, 
            t.arrival_datetime, 
            r.distance_km, 
            t.fuel_consumed_liters, 
            r.toll_cost,
            t.vehicle_id, 
            t.driver_id, 
            t.route_id
        FROM deliveries d
        JOIN trips t ON d.trip_id = t.trip_id
        JOIN routes r ON t.route_id = r.route_id
        WHERE d.scheduled_datetime::date = CURRENT_DATE - INTERVAL '1 day'
        """
        
        try:
            df = pd.read_sql(query, self.pg_conn)
            self.metrics['records_extracted'] = len(df)
            logging.info(f" Extraídos {len(df)} registros")
            return df
        except Exception as e:
            logging.error(f" Error en extracción: {e}")
            self.metrics['errors'] += 1
            return pd.DataFrame()

### **2.5. Transformación de Datos**

In [ ]:
# Función para transformación de datos en Python. 
# Se realizan algunas operaciones matemáticas para obtener las máetricas
    
    def transform_data(self, df: pd.DataFrame) -> pd.DataFrame:
        """Transformar datos para el modelo dimensional"""
        logging.info(" Iniciando transformación de datos...")
        
        try:
            # Calcular métricas

            # Tiempo de entrega en minutos
            df['delivery_time_minutes'] = (
                (pd.to_datetime(df['delivered_datetime']) - 
                 pd.to_datetime(df['scheduled_datetime'])).dt.total_seconds() / 60
            ).round(2)

            # Tiempo de retraso en minutos
            df['delay_minutes'] = df['delivery_time_minutes'].apply(
                lambda x: max(0, x) if x > 0 else 0
            )

            # Está a tiempo si la entrega se entregó con tiempo menor o igual a 30 minutos
            df['is_on_time'] = df['delay_minutes'] <= 30
            
            # Calcular entregas por hora: división entre 3600s
            df['trip_duration_hours'] = (
                (pd.to_datetime(df['arrival_datetime']) - 
                 pd.to_datetime(df['departure_datetime'])).dt.total_seconds() / 3600
            ).round(2)
            
            # Agrupar entregas por trip para calcular entregas/hora. Se efectúa división
            deliveries_per_trip = df.groupby('trip_id').size()
            df['deliveries_in_trip'] = df['trip_id'].map(deliveries_per_trip)
            df['deliveries_per_hour'] = (
                df['deliveries_in_trip'] / df['trip_duration_hours']
            ).round(2)
            
            # Eficiencia de combustible. División de distancia entre litros consumidos de combustible
            df['fuel_efficiency_km_per_liter'] = (
                df['distance_km'] / df['fuel_consumed_liters']
            ).round(2)
            
            # Costo estimado por entrega. Se multiplica los litros consumidos de combustiblle por un 
            # estimado de precio de 5000 y se le adiciona el costo total que incluye otros costos operativos, 
            # luego se divide entre el número de entregas por viaje 
            df['cost_per_delivery'] = (
                (df['fuel_consumed_liters'] * 5000 + df['toll_cost']) / 
                df['deliveries_in_trip']
            ).round(2)
            
            # Revenue estimado (ejemplo: $20,000 base + $500 por kg)
            df['revenue_per_delivery'] = (20000 + df['package_weight_kg'] * 500).round(2)
            
            # Validaciones de calidad
            # No permitir tiempos negativos
            df = df[df['delivery_time_minutes'] >= 0]
            
            # No permitir pesos fuera de rango
            df = df[(df['package_weight_kg'] > 0) & (df['package_weight_kg'] < 10000)]
            
            # Manejar cambios históricos (SCD Type 2 para conductor/vehículo)
            df['valid_from'] = pd.to_datetime(df['scheduled_datetime']).dt.date
            df['valid_to'] = pd.to_datetime('9999-12-31')
            df['is_current'] = True
            
            self.metrics['records_transformed'] = len(df)
            logging.info(f" Transformados {len(df)} registros")
            
            return df
            
        except Exception as e:
            logging.error(f" Error en transformación: {e}")
            self.metrics['errors'] += 1
            return pd.DataFrame()

### **2.6. Carga de Datos**

In [ ]:
# Función de carga de datos. 
    
    def load_dimensions(self, df: pd.DataFrame):
        """Cargar o actualizar dimensiones en Snowflake"""
        logging.info(" Cargando dimensiones...")
        
        cursor = self.sf_conn.cursor()
        
        try:
            # Cargar dim_customer (nuevos clientes)
            customers = df[['customer_name']].drop_duplicates()
            for _, row in customers.iterrows():
                cursor.execute("""
                    MERGE INTO dim_customer c
                    USING (SELECT %s as customer_name) s
                    ON c.customer_name = s.customer_name
                    WHEN NOT MATCHED THEN
                        INSERT (customer_name, customer_type, city, first_delivery_date, 
                               total_deliveries, customer_category)
                        VALUES (%s, 'Individual', %s, CURRENT_DATE(), 0, 'Regular')
                """, (row['customer_name'], row['customer_name'], 
                     df[df['customer_name'] == row['customer_name']]['destination_city'].iloc[0]))
            
            # Actualizar dimensiones SCD Type 2 si hay cambios
            # (Ejemplo simplificado para dim_driver)
            cursor.execute("""
                UPDATE dim_driver 
                SET valid_to = CURRENT_DATE() - 1, is_current = FALSE
                WHERE driver_id IN (
                    SELECT DISTINCT driver_id 
                    FROM staging_daily_load
                ) AND is_current = TRUE
            """)
            
            self.sf_conn.commit()
            logging.info(" Dimensiones actualizadas")
            
        except Exception as e:
            logging.error(f" Error cargando dimensiones: {e}")
            self.sf_conn.rollback()
            self.metrics['errors'] += 1
    
    def load_facts(self, df: pd.DataFrame):
        """Cargar hechos en Snowflake"""
        logging.info(" Cargando tabla de hechos...")
        
        cursor = self.sf_conn.cursor()
        
        try:
            # Preparar datos para inserción
            fact_data = []
            for _, row in df.iterrows():
                # Obtener keys de dimensiones
                date_key = int(pd.to_datetime(row['scheduled_datetime']).strftime('%Y%m%d'))
                scheduled_time_key = pd.to_datetime(row['scheduled_datetime']).hour * 100
                delivered_time_key = pd.to_datetime(row['delivered_datetime']).hour * 100
                
                fact_data.append((
                    date_key,
                    scheduled_time_key,
                    delivered_time_key,
                    row['vehicle_id'],  # Simplificado, debería buscar vehicle_key
                    row['driver_id'],   # Simplificado, debería buscar driver_key
                    row['route_id'],    # Simplificado, debería buscar route_key
                    1,  # customer_key placeholder
                    row['delivery_id'],
                    row['trip_id'],
                    row['tracking_number'],
                    row['package_weight_kg'],
                    row['distance_km'],
                    row['fuel_consumed_liters'],
                    row['delivery_time_minutes'],
                    row['delay_minutes'],
                    row['deliveries_per_hour'],
                    row['fuel_efficiency_km_per_liter'],
                    row['cost_per_delivery'],
                    row['revenue_per_delivery'],
                    row['is_on_time'],
                    False,  # is_damaged
                    row['recipient_signature'],
                    row['delivery_status'],
                    self.batch_id
                ))
            
            cursor.executemany("""
                INSERT INTO fact_deliveries (
                    date_key, scheduled_time_key, delivered_time_key,
                    vehicle_key, driver_key, route_key, customer_key,
                    delivery_id, trip_id, tracking_number,
                    package_weight_kg, distance_km, fuel_consumed_liters,
                    delivery_time_minutes, delay_minutes, deliveries_per_hour,
                    fuel_efficiency_km_per_liter, cost_per_delivery, revenue_per_delivery,
                    is_on_time, is_damaged, has_signature, delivery_status,
                    etl_batch_id
                ) VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
            """, fact_data)
            
            self.sf_conn.commit()
            self.metrics['records_loaded'] = len(fact_data)
            logging.info(f" Cargados {len(fact_data)} registros en fact_deliveries")
            
        except Exception as e:
            logging.error(f" Error cargando hechos: {e}")
            self.sf_conn.rollback()
            self.metrics['errors'] += 1


### **2.7. Ejecución de Pipeline**

Tiene dos modificaciones del código para que funcionará

In [ ]:
# Función para ejecutar el pipeline desde extracción, pasando por transformación y terminando en cargado de datos.

    def run_etl(self):
        """Ejecutar pipeline ETL completo"""
        start_time = datetime.now()
        logging.info(f" Iniciando ETL - Batch ID: {self.batch_id}")
        
        try:
            # Conectar
            if not self.connect_databases():
                return
            
            # ETL
            df = self.extract_daily_data()
            if not df.empty:
                df_transformed = self.transform_data(df)
                if not df_transformed.empty:
                    self.load_dimensions(df_transformed)
                    self.load_facts(df_transformed)
            
            # Calcular totales para reportes
            self._calculate_daily_totals()
            
            # Cerrar conexiones
            self.close_connections()
            
            # Log final
            duration = (datetime.now() - start_time).total_seconds()
            logging.info(f" ETL completado en {duration:.2f} segundos")
            logging.info(f" Métricas: {json.dumps(self.metrics, indent=2)}")
            
        except Exception as e:
            logging.error(f" Error fatal en ETL: {e}")
            self.metrics['errors'] += 1
            self.close_connections()
    
    def _calculate_daily_totals(self):
        """Pre-calcular totales para reportes rápidos"""
        cursor = self.sf_conn.cursor()
        
        try:

# Se modificó esta parte para crear la tabla de totales. 
# Contiene métricas obtenidas en la etapa de transformación de datos

            cursor.execute("""
                CREATE TABLE IF NOT EXISTS daily_delivery_totals (
                    date_key INT REFERENCES dim_date(date_key),
                    etl_batch_id INT,
                    total_deliveries INT,
                    on_time_deliveries INT,
                    total_weight_kg DECIMAL(10,2),
                    total_distance_km DECIMAL(10,2),
                    total_fuel_liters DECIMAL(10,2),
                    avg_delivery_time_minutes DECIMAL(10,2),
                    total_cost DECIMAL(10,2),
                    total_revenue DECIMAL(10,2),
                    created_at TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
                )
            """)
            
# Se modificó esta parta para insertar totales diarios y realizar las operaciones de suma y promedio 
# de las métricas con los datos previamente insertados en fact_deliveries. 
            cursor.execute("""
                INSERT INTO daily_delivery_totals (
                    date_key,
                    etl_batch_id,
                    total_deliveries,
                    on_time_deliveries,
                    total_weight_kg,
                    total_distance_km,
                    total_fuel_liters,
                    avg_delivery_time_minutes,
                    total_cost,
                    total_revenue
                )
                SELECT 
                    date_key,
                    etl_batch_id,
                    COUNT(*) AS total_deliveries,
                    COUNT_IF(is_on_time = TRUE) AS on_time_deliveries,
                    SUM(package_weight_kg) AS total_weight_kg,
                    SUM(distance_km) AS total_distance_km,
                    SUM(fuel_consumed_liters) AS total_fuel_liters,
                    AVG(delivery_time_minutes) AS avg_delivery_time_minutes,
                    SUM(cost_per_delivery) AS total_cost,
                    SUM(revenue_per_delivery) AS total_revenue
                FROM fact_deliveries
                WHERE etl_batch_id = %s
                GROUP BY date_key, etl_batch_id
            """, (self.batch_id,))
            
            self.sf_conn.commit()
            logging.info(" Totales diarios calculados")
            
        except Exception as e:
            logging.error(f" Error calculando totales: {e}")

### **2.8. Cierre de Conexiones**

In [ ]:
# Se realiza el respectivo cierre de conexiones a la fuente y recepción de datos
    def close_connections(self):
        if self.pg_conn:
            self.pg_conn.close()
        if self.sf_conn:
            self.sf_conn.close()
        logging.info(" Conexiones cerradas")

### **2.9. Automatización**

In [ ]:
# Se programa funciones para que el sistema actualice las bases de datos en Snowflake 

def job():
    """Función para programar con schedule"""
    etl = FleetLogixETL()
    etl.run_etl()

def main():
    """Función principal - Automatización diaria"""
    logging.info(" Pipeline ETL FleetLogix iniciado")
    
    # Programar ejecución diaria a las 2:00 AM
    schedule.every().day.at("02:00").do(job)
    
    logging.info(" ETL programado para ejecutarse diariamente a las 2:00 AM")
    logging.info("Presiona Ctrl+C para detener")
    
    # Ejecutar una vez al inicio (para pruebas)
    job()
    
    # Loop infinito esperando la hora programada
    while True:
        schedule.run_pending()
        time.sleep(60)  # Verificar cada minuto

if __name__ == "__main__":
    main()